# Models, Context, Tools and Skills

Notebook 01 showed that the machinery around a model decides the result: the same model could not produce the answer with a bare prompt, nor with a better prompt, and produced it with a cited source inside a harness. This notebook opens the first four pieces of that machinery one at a time.

- The **model** sits behind one small interface, so it can be swapped: a real Claude model or a deterministic stand-in.
- The **context** is everything the model sees in one call, and it is a budget, not a bucket.
- **Tools** are functions the model can ask for, and how you describe them is prompt engineering for machines.
- **Skills** are folders of instructions the agent opens only when a task needs them.

**What you will learn**

- What the whole contract between a harness and a model looks like, in six lines of Python, and what `model_for` puts behind it
- How to estimate tokens, watch a long session run out of budget, and see compaction fold it back
- How a tool turns into the JSON schema a model reads, why a vague description hurts, and how errors and permissions belong to the interface
- How progressive disclosure keeps a library of skills almost free until one is needed, on the stand-in and on Claude

## How to run this notebook

You can run this notebook in two ways:

- **Locally with Jupyter:** open it in Jupyter Notebook, JupyterLab or VS Code and run the cells top to bottom.
- **On Google Colab:** upload the file at [colab.research.google.com](https://colab.research.google.com) (File > Upload notebook) and run the cells top to bottom.

**Which engine drives the harness.** The notebook runs on **real Claude models by default** (`claude-opus-5` for the main agents) when it finds `ANTHROPIC_API_KEY` in a git-ignored `.env` file at the repository root (copy `.env.example`). Without a key it falls back to the **ScriptedModel**, a small Python function that plays the model's part deterministically, and every cell still runs.

A few practical notes:

- **Never paste a key into a cell.** The repository's checks reject notebooks that contain anything that looks like one. On Colab, store the key in the Secrets panel and load it into the environment before the setup cell, or run without a key.
- Real runs **cost money and vary between runs**; this notebook makes only a handful of real model calls. The compaction session, the skill token counts and the budget exercise are **reference runs** on the stand-in, so their numbers are the same every time.
- The **setup cell** below downloads the course library (a folder called `harness`) when it is not next to the notebook, as on Colab. Locally the folder is already there.
- Section 2 and the exercises write small files to an `outputs/` folder next to the notebook. They are scratch files; you can delete the folder at any time.

In [1]:
# Setup: the notebooks use a small teaching library called `harness` that lives next to them
# in the repository. In Google Colab the folder is not there, so we download it (public files,
# no key, no account). Locally and in CI it is already present.
import pathlib, sys, urllib.request

REPO = "https://raw.githubusercontent.com/maglionejm/fontys-tech-exercises/main/M5%20-%20Harness%20Engineering/"
FILES = [
    "harness/__init__.py",
    "harness/agent.py",
    "harness/context.py",
    "harness/corpus/agent-skills.md",
    "harness/corpus/context-engineering.md",
    "harness/corpus/evals-and-observability.md",
    "harness/corpus/harness-effect.md",
    "harness/corpus/model-context-protocol.md",
    "harness/corpus/multi-agent-systems.md",
    "harness/corpus/prompt-to-harness-timeline.md",
    "harness/corpus/tool-design.md",
    "harness/corpus/what-is-a-harness.md",
    "harness/corpus/workflow-patterns.md",
    "harness/demo_tools.py",
    "harness/evals.py",
    "harness/messages.py",
    "harness/models.py",
    "harness/multi.py",
    "harness/patterns.py",
    "harness/policies.py",
    "harness/providers.py",
    "harness/runtime.py",
    "harness/skills.py",
    "harness/skills/cite-sources/SKILL.md",
    "harness/skills/cite-sources/checklist.md",
    "harness/skills/safe-calculations/SKILL.md",
    "harness/skills/summarize-brief/SKILL.md",
    "harness/skills/summarize-brief/template.md",
    "harness/tools.py",
    "harness/trace.py",
]
if not pathlib.Path("harness/__init__.py").exists():
    for name in FILES:
        target = pathlib.Path(name); target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(REPO + name.replace(" ", "%20"), target)
    print(f"Downloaded the harness library ({len(FILES)} files).")
sys.path.insert(0, ".")
import harness
print("harness", harness.__version__, "ready")

# Which engine runs today? A real Claude model when ANTHROPIC_API_KEY is available (loaded from a
# git-ignored .env file at the repository root), otherwise the deterministic ScriptedModel stand-in.
# On Colab the anthropic package is not preinstalled, so install it only when a key is present.
import importlib.util, subprocess
if harness.real_model_available() and importlib.util.find_spec("anthropic") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "anthropic"])

harness 0.2.1 ready


In [2]:
from harness import describe_runtime

print(describe_runtime())

Engine: real Claude models - claude-opus-5 for lead agents, claude-sonnet-5 for workers and judges (key loaded from .env; the key is never printed).


Without a key the notebook runs on the ScriptedModel stand-in and every number is reproducible; with a key it runs on Claude and numbers vary between runs. Never put a key in a cell; use `.env` (see `.env.example` at the repository root).

## 1. The model behind an interface

> A **model** is the text-in, text-out engine. In this module, anything that maps a context to the next message. Our **ScriptedModel** is a stand-in that shows the mechanics without an API key.

A harness never needs to know what a model is made of. It needs exactly one thing from it: given the system prompt, the conversation so far and the tools on offer, return the next message. The course library writes that contract down as a **protocol**, a Python type that lists the methods an object must have. Here it is, straight from the source file.

In [3]:
import inspect
from harness.models import Model

print(inspect.getsource(Model))

class Model(Protocol):
    """The whole contract between a harness and a model."""

    name: str

    def complete(self, system: str, messages: list[Message],
                 tools: list[dict[str, Any]]) -> Message: ...



That is the whole interface: a `name` and one method, `complete`. Anything with those two things can be the engine of our harness.

The `ScriptedModel` implements it with a **policy**, a Python function that reads what the model would see and decides the next message. We ask for the stand-in explicitly here with `model_for("bare", real=False)`, and call it directly, without any agent loop around it, to see the contract in action: context in, next message out.

In [4]:
from harness import model_for, Message

stand_in, _ = model_for("bare", real=False)
reply = stand_in.complete(system="",
                          messages=[Message("user", "When did Anthropic open-source the Model Context Protocol?")],
                          tools=[])
print(type(stand_in).__name__, stand_in.name)
print(reply)
print("calls made so far:", stand_in.calls_made)

ScriptedModel scripted-bare
Message(role='assistant', content='Anthropic released the Model Context Protocol in 2023 and Google adopted it in 2024.', tool_calls=[], tool_call_id=None, name=None, raw=None, usage=None)
calls made so far: 1


What does a policy get to look at? Exactly what a real model would see, wrapped in a `ModelView` with a few helpers: the current task, the tools on offer, which tools have already been called for this task and what they returned. Nothing else. A policy cannot peek at the corpus or the harness; it can only ask through tools, like a real model.

In [5]:
from harness import ModelView
from harness.demo_tools import search_docs, read_doc

view = ModelView(system="Answer from the library.",
                 messages=[Message("user", "When was the Model Context Protocol released?")],
                 tools=[search_docs.schema(), read_doc.schema()])

print("task:           ", view.task)
print("tools on offer: ", view.tool_names())
print("has search_docs:", view.has_tool("search_docs"))
print("called it yet:  ", view.called("search_docs"))

task:            When was the Model Context Protocol released?
tools on offer:  ['search_docs', 'read_doc']
has search_docs: True
called it yet:   False


Writing a policy is writing a caricature of one model behaviour. The helpers `say` (a final text reply) and `call` (a request to run one tool) build the two kinds of message a model can return.

In [6]:
from harness import ScriptedModel
from harness.models import say, call


def one_search_policy(view):
    """Look once, then report what came back."""
    if view.has_tool("search_docs") and not view.called("search_docs"):
        return call("search_docs", query="model context protocol", k=1)
    return say("The library says: " + (view.last_result("search_docs") or "nothing")[:100])


first = ScriptedModel(one_search_policy, name="one-search").complete(
    system="", messages=[Message("user", "When was MCP released?")], tools=[search_docs.schema()])
print(first)

Message(role='assistant', content='', tool_calls=[ToolCall(id='call_0001', name='search_docs', arguments={'query': 'model context protocol', 'k': 1})], tool_call_id=None, name=None, raw=None, usage=None)


The first message is a tool call, not an answer. In a harness, the loop would now run `search_docs`, append the result as a tool message, and call `complete` again; the policy would then reach its second branch and answer. Notebook 01 built exactly that loop.

**A real model is a drop-in.** The library's `providers.py` wraps the official Anthropic and OpenAI SDKs behind the same `complete` method. Notice that `complete` takes the same three arguments as the protocol above. The adapter's job is translation: our `Message` objects to the provider's request format and back, plus bookkeeping of the real token usage the API reports.

In [7]:
from harness.providers import AnthropicModel, OpenAIModel

print("AnthropicModel", inspect.signature(AnthropicModel.__init__))
print("  .complete    ", inspect.signature(AnthropicModel.complete))
print("OpenAIModel   ", inspect.signature(OpenAIModel.__init__))
print("  .complete    ", inspect.signature(OpenAIModel.complete))

AnthropicModel (self, model: 'str' = 'claude-opus-5', api_key: 'str | None' = None, max_tokens: 'int' = 4096, effort: 'str' = 'medium')
  .complete     (self, system: 'str', messages: 'list[Message]', tools: 'list[dict[str, Any]]') -> 'Message'
OpenAIModel    (self, model: 'str' = 'gpt-5', api_key: 'str | None' = None)
  .complete     (self, system: 'str', messages: 'list[Message]', tools: 'list[dict[str, Any]]') -> 'Message'


This is what `model_for(role)` hands out when a key is present: an `AnthropicModel` for the role's tier, together with the system prompt written for that role. Without a key you get the stand-in and the same prompt. Let us look at what the `research` role gets today.

In [8]:
model, system = model_for("research")
print(f"engine for 'research': {type(model).__name__} named {model.name!r}")
print()
print("system prompt for the role:")
print(system)

engine for 'research': AnthropicModel named 'claude-opus-5'

system prompt for the role:
You answer questions using ONLY the course library, through your tools. Steps: call search_docs with two to six key terms (never the whole question); call read_doc on the best match; draft one or two sentences taken from the document; if a check_citation tool exists, call it with your draft AND the question, and fix whatever it reports; end your final answer with the tag [source: <doc id>]. If the library holds nothing relevant, say exactly that instead of guessing. Keep answers short.


Because the engine is interchangeable, choosing one becomes an engineering decision rather than a fixed fact: pick by **capability**, **cost** and **latency**, and route cheap models to easy steps. The library does this by tier: main roles get `claude-opus-5`, helper roles such as workers, judges and routers get `claude-sonnet-5`. Anthropic's multi-agent research system does the same: a large model plans and synthesizes while smaller, cheaper models do the parallel legwork (Anthropic, "How we built our multi-agent research system", 13 Jun 2025). Notebook 03 builds a router that sends each task to the right agent.

## 2. Context is a budget

> **Context** is everything the model sees in one call: the system prompt, instruction files, the conversation so far, tool results, retrieved documents and notes. **Context engineering** is deciding what goes in, what stays out, and when to summarize.

In June 2025 the industry stopped saying "prompt engineering" and started saying "context engineering": "the delicate art and science of filling the context window with just the right information for the next step" (Andrej Karpathy, 25 Jun 2025). The reason is practical. A model's context window is finite, and models perform worse as it fills with material that is not needed. Anthropic's "Effective context engineering for AI agents" (29 Sep 2025) names three techniques for long tasks, and the course library implements two of them:

- **Compaction**: when the conversation nears its limit, summarize the older part into one note and continue with the summary plus the most recent turns.
- **Structured note-taking**: write facts to a file *outside* the window, so they survive compaction and restarts.
- **Sub-agent architectures**: give each focused task a fresh context and let it return a short summary (notebook 04).

One more piece of context you will meet in every real harness: **instruction files** such as `CLAUDE.md` or `AGENTS.md`, standing rules loaded into every session. Marmelab's 2026 survey found that human-written instruction files improved success by about four percent, while machine-generated ones did worse than no file at all. When everything is marked important, nothing is (Marmelab, "The State of AI Harness Engineering 2026", 24 Sep 2026).

### 2.1 Counting tokens

> A **token** is the unit models read and bill by, about four characters of English text. The library estimates tokens with that rule: the counts are approximate but identical every run.

In [9]:
from harness import estimate_tokens, total_tokens, Agent, Message, model_for
from harness.demo_tools import search_docs, read_doc, check_citation

for text in ["The model is the engine; the harness is the rest of the car.",
             read_doc.call({"doc_id": "model-context-protocol"})]:
    print(f"{estimate_tokens(text):>5} tokens  <- {text[:60]!r}...")

model, system = model_for("research")
research = Agent(model, tools=[search_docs, read_doc, check_citation], system=system, name="research")
print(f"{estimate_tokens(research.system_prompt()):>5} tokens  <- the research agent's system prompt (role instructions + tool descriptions)")

   15 tokens  <- 'The model is the engine; the harness is the rest of the car.'...
  271 tokens  <- 'id: model-context-protocol\ntitle: The Model Context Protocol'...
  252 tokens  <- the research agent's system prompt (role instructions + tool descriptions)


A single library document costs about as much as the whole system prompt, role instructions and tool descriptions included, and every tool result stays in the conversation for every later call. That is why a long session grows quickly.

### 2.2 A five-question session on a small budget

The `ContextWindow` class is a token budget with a compaction rule. Before each model call the agent checks whether the system prompt plus the conversation exceed `budget_tokens`. If they do, it keeps the first message (the original task) and the last few turns, and folds everything in between into one summary note.

We give one research agent a budget of 2,500 tokens and ask it five questions in a row, passing the growing history from one run to the next, as a chat application would. This is a **reference run on the stand-in** (`model_for("research", real=False)`, with the short prompt the course numbers were recorded with), so the numbers are the same every time and the five questions cost nothing. Watch three numbers per question: how many compactions have happened, how many messages the history holds, and how many tokens the window carries. The last column shows whether the answer still carries a source.

In [10]:
from harness import ContextWindow

QUESTIONS = [
    "When did Anthropic open-source the Model Context Protocol?",
    "Which company adopted the Model Context Protocol in March 2025?",
    "When did Anthropic introduce Agent Skills?",
    "How many levels does progressive disclosure have in Agent Skills?",
    "By how much did Anthropic's multi-agent research system beat a single agent?",
]

session = Agent(model_for("research", real=False)[0], tools=[search_docs, read_doc, check_citation],
                system="Answer from the library and cite.",
                context=ContextWindow(budget_tokens=2500), name="session")
history = []
for i, q in enumerate(QUESTIONS, 1):
    result = session.run(q, history=history)
    history = result.messages
    window = session.context.tokens(session.system_prompt(), history)
    print(f"Q{i}: compactions={session.context.compactions}  messages={len(history):>2}  "
          f"window~{window:>4} tokens  cited={'[source:' in result.answer}")
    print(f"    {result.answer[:95]}...")

Q1: compactions=0  messages= 8  window~ 786 tokens  cited=True
    Anthropic open-sourced the Model Context Protocol on 25 November 2024, with launch partners inc...
Q2: compactions=0  messages=16  window~1438 tokens  cited=True
    On 26 March 2025 OpenAI announced support for MCP in its Agents SDK, with the ChatGPT desktop a...
Q3: compactions=0  messages=24  window~2120 tokens  cited=True
    Anthropic introduced Agent Skills on 16 October 2025 in the post "Equipping agents for the real...
Q4: compactions=1  messages=11  window~1342 tokens  cited=True
    Skills work through progressive disclosure, in three levels. [source: agent-skills]...
Q5: compactions=1  messages=19  window~2283 tokens  cited=True
    Anthropic's post "How we built our multi-agent research system" (13 June 2025) describes an orc...


The history grows by eight messages per question (the question, three tool calls, three tool results, the answer). By the end of the third question the window is close to the budget. During the fourth question it crosses it, and the agent compacts before its next model call: the message count drops from 24 to 11, the token count drops with it, and the session carries on. Every answer stays cited, because the current task and its fresh tool results are never summarized away.

What does the summary look like? It is a plain message in the history, marked so the model (and the policy) can tell it apart from a real user turn.

In [11]:
notes_in_history = [m for m in history if m.content.startswith("[Context compacted")]
print(f"{len(notes_in_history)} compaction note in the history. Its text:\n")
print(notes_in_history[0].content)

1 compaction note in the history. Its text:

[Context compacted: 22 earlier messages summarized]
- called search_docs(query='anthropic open-source model context prot', k='3')
- search_docs returned: [{"id": "model-context-protocol", "title": "The Model Context Protocol", "score": 13.07, "snippet": "The Model Context P
- called read_doc(doc_id='model-context-protocol')
- read_doc returned: id: model-context-protocol
- called check_citation(answer='Anthropic open-sourced the Model Context', question='When did Anthropic open-source the Model')
- check_citation returned: OK: 1 citation(s) checked against the library.
- said: Anthropic open-sourced the Model Context Protocol on 25 November 2024, with launch partners including Block and Apollo and development tool makers Zed, Replit, 
- user: Which company adopted the Model Context Protocol in March 2025?
- called search_docs(query='company adopted model context protocol m', k='3')
- search_docs returned: [{"id": "model-context-protocol", "t

The summary is deterministic here: which tools ran, the first line of what came back, what was said. A real harness asks the model to write the summary, and the quality of that summary (keep decisions and open problems, drop redundant tool output) is one of the things context engineering is about.

### 2.3 Notes that live outside the window

Compaction is lossy. A fact buried in a summarized tool result may be gone for good. **Structured note-taking**, also called agentic memory, writes the facts that matter to a file outside the context window. The library provides a `Notes` file and two tools around it, `write_note` and `read_notes`, that an agent can call like any other tool. We call them directly here.

In [12]:
from harness import Notes
from harness.demo_tools import write_note, read_notes

Notes("outputs/notes.md").clear()                 # start from an empty notes file every run

print(write_note.call({"text": "MCP open-sourced 25 November 2024 [source: model-context-protocol]"}))
print(write_note.call({"text": "Agent Skills introduced 16 October 2025 [source: agent-skills]"}))
print()
print("read_notes returns:")
print(read_notes.call({}))
print(f"The notes file holds {Notes('outputs/notes.md').tokens()} tokens, and none of them sit in the context window.")

Noted (16 tokens).
Noted (15 tokens).

read_notes returns:
MCP open-sourced 25 November 2024 [source: model-context-protocol]
Agent Skills introduced 16 October 2025 [source: agent-skills]

The notes file holds 32 tokens, and none of them sit in the context window.


Two facts, a few dozen tokens, and they survive any number of compactions or a fresh session. `write_note` carries the risk level `write` because it changes something outside the conversation; section 3 explains what that risk level is for.

## 3. Tools are prompt engineering for machines

> A **tool** is a function the model can ask for by name, with JSON arguments; the harness runs it and pastes the result back.

The model never runs anything. It reads a list of tool descriptions, decides to ask for one, and receives whatever text the harness sends back. So the description is the whole interface: it is the only thing the model knows about the tool. Anthropic's "Writing effective tools for agents" (Sep 2025) treats tool design as prompt engineering for machines: say when to use the tool and what it returns; group related tools under a shared prefix (namespacing); return meaningful context, not raw dumps; keep results token-efficient by filtering, truncating and paginating.

### 3.1 From a Python function to a JSON schema

The library's `@tool` decorator turns an ordinary function into a `Tool`. It reads the signature for the argument names and types, the first paragraph of the docstring for the description, and lines of the form `argument: explanation` for the argument descriptions. The result is a **JSON schema**, a machine-readable description of the arguments, which is exactly what the Anthropic and OpenAI APIs expect. Here is what `search_docs` looks like from the model's side.

In [13]:
import json
from harness import tool, ToolRegistry, estimate_tokens
from harness.demo_tools import search_docs, read_doc

print(json.dumps(search_docs.schema(), indent=2))

{
  "name": "search_docs",
  "description": "Search the course library for documents about a topic. Returns the best matches as JSON with id, title, score and a snippet; call read_doc with an id to get the full text.",
  "parameters": {
    "type": "object",
    "properties": {
      "query": {
        "type": "string",
        "description": "the key terms to look for, a few words, no full sentences"
      },
      "k": {
        "type": "integer",
        "default": 3,
        "description": "how many results to return, default 3"
      }
    },
    "required": [
      "query"
    ]
  }
}


### 3.2 A vague tool next to a clear one

Here is the same search function described badly. The decorator lets us override the name and description, so we can build a tool called `lookup` whose entire description is "Searches." The function underneath is identical; only what the model reads changes.

In [14]:
@tool(name="lookup", description="Searches.")
def lookup(q: str) -> str:
    return search_docs.fn(q)


for t in (lookup, search_docs):
    print(f"--- {t.name}: {estimate_tokens(t.description)} tokens of description")
    print(ToolRegistry([t]).describe())
    print("arguments:", json.dumps(t.parameters["properties"]))
    print()

--- lookup: 2 tokens of description
- lookup: Searches.
arguments: {"q": {"type": "string"}}

--- search_docs: 42 tokens of description
- search_docs: Search the course library for documents about a topic. Returns the best matches as JSON with id, title, score and a snippet; call read_doc with an id to get the full text.
arguments: {"query": {"type": "string", "description": "the key terms to look for, a few words, no full sentences"}, "k": {"type": "integer", "default": 3, "description": "how many results to return, default 3"}}



Reading `lookup`, a model has to guess what is searched (the web? a database?), what `q` should contain (a sentence? keywords?), what comes back, and what to do next. Reading `search_docs`, it knows all four: the course library, a few key terms, JSON with ids and snippets, and "call `read_doc` with an id". The clear description costs a few dozen tokens more and saves whole wasted turns.

Two field results show how much this matters. When Anthropic launched its web search tool, the model kept appending the year 2025 to every query; rewriting the tool description fixed it (Anthropic, "Writing effective tools for agents", Sep 2025). And fewer tools often beat more: in 2026 Vercel removed eighty percent of the tools from one of its agents, success rose from 80% to 100%, token use halved and latency fell from 724 to 141 seconds (Marmelab, "The State of AI Harness Engineering 2026", 24 Sep 2026). Every tool description sits in the context on every call, so an unused tool is not free.

### 3.3 Errors are part of the interface

When a tool fails, the model reads the failure. A Python traceback tells it nothing useful; a sentence that says what went wrong and what to try instead lets it recover in the next turn. The library's tools follow this rule, and the `Tool.call` wrapper catches wrong arguments and unexpected exceptions and turns them into text. Try three kinds of mistake a model can make.

In [15]:
from harness import ToolCall

print("1. a document id that does not exist:")
print("  ", read_doc.call({"doc_id": "harness-101"}))

print("2. the right tool, the wrong argument name:")
print("  ", read_doc.call({"id": "model-context-protocol"}))

print("3. a tool that is not in the registry:")
print("  ", ToolRegistry([search_docs, read_doc]).call(ToolCall("c1", "fetch_url", {"url": "https://example.org"})))

1. a document id that does not exist:
   Error: no document with id 'harness-101'. Use search_docs to find ids.
2. the right tool, the wrong argument name:
   Error: bad arguments for read_doc: read_doc() got an unexpected keyword argument 'id'. Expected ['doc_id'].
3. a tool that is not in the registry:
   Error: unknown tool 'fetch_url'. Available tools: search_docs, read_doc.


Each message names the problem and points to the fix: use `search_docs` to find ids, here is the list of expected arguments, here are the tools that do exist. A model that reads these can correct itself; a model that reads `KeyError: 'harness-101'` usually cannot.

### 3.4 Risk levels and permissions

> A **guardrail** is a check that runs outside the model.

Some tools only read (a search), some write (a note), and some can do real damage (run arbitrary code, delete files, send money). The library gives every tool a **risk level**: `read`, `write` or `danger`. The agent's `permission` rule is a function that looks at the tool and says yes or no *before* the tool runs, and the default rule refuses anything marked `danger`. This is the harness equivalent of the tool safeguards with risk ratings in OpenAI's guide to building agents (OpenAI, "A practical guide to building agents", Apr 2025), and of the approval prompts you see in Claude Code.

To test it we write a deliberately hostile policy: a stand-in model that, given a `run_python` tool, immediately tries to run code with it. This stays a `ScriptedModel` whatever engine the notebook uses; a guardrail must be tested against a model that misbehaves on purpose, and we do not ask Claude to play that part.

In [16]:
from harness import Agent, ScriptedModel, allow_all
from harness.models import call, say
from harness.demo_tools import run_python, write_note, check_citation

for t in (search_docs, read_doc, check_citation, write_note, run_python):
    print(f"  {t.name:<16} risk = {t.risk}")


def hacker(view):
    if view.has_tool("run_python") and not view.called("run_python"):
        return call("run_python", code="print('pwned')")
    return say("done: " + (view.last_result("run_python") or ""))


print("\nDefault permission (danger is refused):")
guarded = Agent(ScriptedModel(hacker), tools=[run_python], name="guarded")
r_guarded = guarded.run("run code")
print("  answer:", r_guarded.answer)
r_guarded.trace.show()

print("\npermission=allow_all (everything runs):")
open_agent = Agent(ScriptedModel(hacker), tools=[run_python], permission=allow_all, name="open")
print("  answer:", open_agent.run("run code").answer.strip())

  search_docs      risk = read
  read_doc         risk = read
  check_citation   risk = read
  write_note       risk = write
  run_python       risk = danger

Default permission (danger is refused):
  answer: done: Denied: 'run_python' has risk level 'danger' and is not permitted here.
trace of 'guarded': 2 model calls, 0 tool calls, ~98 tokens
   1  model    scripted           run_python({'code': "print('pwned')"})
   1  denied   run_python         Denied: 'run_python' has risk level 'danger' and is not p...
   2  model    scripted           done: Denied: 'run_python' has risk level 'danger' and is...
   2  final    guarded            done: Denied: 'run_python' has risk level 'danger' and is...

permission=allow_all (everything runs):
  answer: done: pwned


Same model, same tool, same request. With the default rule the trace shows a `denied` event, the tool never runs, and the model is told why. With `allow_all` the code executes. The decision was never the model's to make, which is the point of a guardrail: it works even when the model is wrong, tricked or hostile.

A warning from the field: permission prompts only help if someone reads them. Marmelab's 2026 survey found that 93% of permission prompts get approved, the "approval paradox". A rule you can write in code (deny `danger` here, require a review there) is worth more than a prompt a tired human clicks through.

### 3.5 MCP: the standard plug

> **MCP**, the Model Context Protocol, is a standard plug, so any tool server fits any model client.

Our `ToolRegistry` hands tools to one model in one program. In real systems the tools live elsewhere: a database, a ticketing system, a browser, a company's internal API. Before MCP, every harness wrote its own connector for every tool. Anthropic open-sourced the Model Context Protocol on 25 November 2024: an **MCP server** exposes tools, resources and prompts over a common message format, and any **MCP client** (a chat application, a coding agent) can connect to it without custom integration code. OpenAI adopted MCP on 26 March 2025 in its Agents SDK, then in ChatGPT and the Responses API; by 2026 it is the default way harnesses discover and call external tools (Anthropic, "Introducing the Model Context Protocol", Nov 2024; corpus document `model-context-protocol`).

In the car metaphor MCP is the standard trailer hitch: the tool server does not need to know which car will tow it. Everything you learned in this section still applies. An MCP tool is still a name, a description and a JSON schema, and a bad description hurts just as much over a standard plug.

## 4. Skills and progressive disclosure

> A **skill** is a folder with a `SKILL.md` file (name, description, instructions) and optional scripts or reference files; the agent reads the description first and opens the rest only when the task matches.

Tools give a model abilities. Skills teach it procedures: how to use those abilities well for a recurring kind of work. Anthropic introduced Agent Skills on 16 October 2025 ("Equipping agents for the real world with Agent Skills") and published the format as an open standard on 18 December 2025; by 2026 more than forty platforms support it.

The idea that makes skills cheap is **progressive disclosure**, in three levels:

1. **Level 1, at startup:** the agent sees only each skill's name and one-line description, a few dozen tokens per skill. A library of a hundred skills costs almost nothing.
2. **Level 2, when a task matches:** the agent loads the full instructions of that one skill.
3. **Level 3, while executing:** it reads the supporting files or runs the scripts the instructions mention.

In the car, skills are the manuals in the glovebox: always there, opened only when something specific comes up. The course library ships three sample skills in `harness/skills/`. `SkillIndex` reads them and exposes the three levels as three methods. The counts below are file sizes, not model calls, so they are the same on any engine.

In [17]:
from harness import SkillIndex

skills = SkillIndex()
print("skills:", skills.names())
print()
print("Level 1, the catalog the agent sees at startup:")
print(skills.catalog())
print()
print(f"catalog (level 1) = {skills.catalog_tokens()} tokens")
print(f"all three bodies (level 2) = {skills.full_tokens()} tokens, "
      f"{skills.full_tokens() / skills.catalog_tokens():.1f}x the catalog")

skills: ['cite-sources', 'safe-calculations', 'summarize-brief']

Level 1, the catalog the agent sees at startup:
- cite-sources: Answer factual questions from the course library with a [source: id] citation, verified with check_citation before replying.
- safe-calculations: Do every calculation with the calculator tool, never in your head, and show the exact expression you evaluated.
- summarize-brief: Turn one library document into a five-line brief with exactly one concrete number or date per line.

catalog (level 1) = 98 tokens
all three bodies (level 2) = 321 tokens, 3.3x the catalog


Three skills at level 1 fit in under a hundred tokens. Loading all three bodies up front would cost more than three times that, and most of it would be irrelevant to any given task. With progressive disclosure the agent pays for level 2 only when a task matches a description.

In [18]:
print("Level 2, the full instructions of one skill:\n")
print(skills.load("cite-sources"))

Level 2, the full instructions of one skill:

# Skill: cite-sources

Use this skill whenever a question asks for a fact, a date or a number.

1. Search the library with search_docs using two to six key terms, never the whole question.
2. Read the best match with read_doc. If it does not answer the question, read the runner-up.
3. Draft an answer in one or two sentences taken from the document, and end it with the tag
   [source: <doc id>].
4. Call check_citation with the full draft. If it returns PROBLEM, fix the draft or read another
   document; never reply with an unverified claim.
5. Reply with the verified answer. If nothing in the library supports an answer, say so.

See checklist.md for the review checklist teachers apply to cited answers.

Files in this skill: checklist.md


In [19]:
print("Level 3, one supporting file the instructions point to:\n")
print(skills.resource("cite-sources", "checklist.md"))

Level 3, one supporting file the instructions point to:

Cited-answer checklist
- The [source: id] tag names a document that exists in the library.
- Every year or number in the answer appears in that document.
- The answer uses the document's words for the fact, not a paraphrase that changes it.
- If two documents disagree, both are cited and the disagreement is stated.



### 4.1 An agent that can open the glovebox

Pass a `SkillIndex` to an `Agent` and two things happen. The catalog is appended to the system prompt, so the model sees level 1 on every call. And two tools are added to the registry, `load_skill` and `read_skill_file`, so the model can walk down to levels 2 and 3 itself. Look at the end of the system prompt and at the tool list.

In [20]:
from harness import Agent, model_for
from harness.demo_tools import search_docs, read_doc, check_citation

model, system = model_for("research")
skilled_agent = Agent(model, tools=[search_docs, read_doc, check_citation],
                      skills=SkillIndex(), system=system, name="skilled")

print(skilled_agent.system_prompt())
print()
print("tools:", skilled_agent.tools.names())

You answer questions using ONLY the course library, through your tools. Steps: call search_docs with two to six key terms (never the whole question); call read_doc on the best match; draft one or two sentences taken from the document; if a check_citation tool exists, call it with your draft AND the question, and fix whatever it reports; end your final answer with the tag [source: <doc id>]. If the library holds nothing relevant, say exactly that instead of guessing. Keep answers short.

Tools available:
- search_docs: Search the course library for documents about a topic. Returns the best matches as JSON with id, title, score and a snippet; call read_doc with an id to get the full text.
- read_doc: Read one document from the course library in full.
- check_citation: Verify a draft answer before giving it: it must cite a library document as [source: id], the document must contain the answer's key terms and every year in it, and it must address the question. Returns OK or PROBLEM with th

Now run it. The question matches the `cite-sources` description, so a model that reads the catalog should open that skill before it searches. On Claude, watch the trace for a `load_skill` call and read the token column: those are real input tokens reported by the API, and the catalog is part of every one of them. On the stand-in, the research policy does not know about skills and goes straight to the search; the reference run below shows the loading step deterministically.

In [21]:
question = "When did Anthropic introduce Agent Skills?"

r_skilled = skilled_agent.run(question)
print(r_skilled.answer)
print()
r_skilled.trace.show()
print()
print("loaded a skill:", any(e.name == "load_skill" for e in r_skilled.trace.tool_calls()),
      "| model calls:", r_skilled.turns, "| tokens:", r_skilled.tokens)

Anthropic introduced Agent Skills on 16 October 2025 in the post "Equipping agents for the real world with Agent Skills", and published the format as an open standard on 18 December 2025. [source: agent-skills]

trace of 'skilled': 4 model calls, 4 tool calls, ~9277 tokens
   1  model    claude-opus-5      I'll search the course library for this. search_docs({'qu...
   1  tool     search_docs        (query='Anthropic Agent Skills introduced') -> [{"id": "a...
   1  tool     load_skill         (name='cite-sources') -> # Skill: cite-sources  Use this ...
   2  model    claude-opus-5      read_doc({'doc_id': 'agent-skills'})
   2  tool     read_doc           (doc_id='agent-skills') -> id: agent-skills title: Agent ...
   3  model    claude-opus-5      check_citation({'answer': 'Anthropic introduced Agent Ski...
   3  tool     check_citation     (answer='Anthropic introduced Agent Skills on 16 ', quest...
   4  model    claude-opus-5      Anthropic introduced Agent Skills on 16 October 202

A real model may combine steps: in the run recorded here Claude opened the skill and ran its first search in the same turn, then read the document, checked the citation and answered. The token column shows the price of carrying the catalog and the loaded skill through every later call.

### 4.2 Reference run on the stand-in: what level 2 costs

To see the levels in a trace every time, we need a policy that uses them. `skilled_policy` is a stand-in that loads the `cite-sources` skill first, then behaves like the stand-in research policy. We run it next to a plain stand-in research agent on the same question, with the short prompt the course numbers were recorded with, so the difference between the two token counts is exactly the price of skills: the catalog on every call, plus one level-2 load.

In [22]:
from harness import ScriptedModel
from harness.models import call
from harness.policies import research_policy


def skilled_policy(view):
    """Open the matching skill first, then follow the research procedure."""
    if view.has_tool("load_skill") and not view.called("load_skill"):
        return call("load_skill", name="cite-sources")
    return research_policy(view)


plain = Agent(model_for("research", real=False)[0], tools=[search_docs, read_doc, check_citation],
              system="Answer from the library and cite.", name="plain")
with_skill = Agent(ScriptedModel(skilled_policy, "scripted-skilled"), tools=[search_docs, read_doc, check_citation],
                   skills=SkillIndex(), system="Answer from the library and cite.", name="with-skill")

r_plain = plain.run(question)
r_skill = with_skill.run(question)
r_skill.trace.show()
print()
print(f"without skills: {r_plain.turns} model calls, ~{r_plain.tokens} tokens")
print(f"with skills:    {r_skill.turns} model calls, ~{r_skill.tokens} tokens")
print(f"extra cost of the catalog on every call plus one level-2 load: ~{r_skill.tokens - r_plain.tokens} tokens")

trace of 'with-skill': 5 model calls, 4 tool calls, ~3684 tokens
   1  model    scripted-skilled   load_skill({'name': 'cite-sources'})
   1  tool     load_skill         (name='cite-sources') -> # Skill: cite-sources  Use this ...
   2  model    scripted-skilled   search_docs({'query': 'anthropic introduce agent skills',...
   2  tool     search_docs        (query='anthropic introduce agent skills', k='3') -> [{"i...
   3  model    scripted-skilled   read_doc({'doc_id': 'agent-skills'})
   3  tool     read_doc           (doc_id='agent-skills') -> id: agent-skills title: Agent ...
   4  model    scripted-skilled   check_citation({'answer': 'Anthropic introduced Agent Ski...
   4  tool     check_citation     (answer='Anthropic introduced Agent Skills on 16 ', quest...
   5  model    scripted-skilled   Anthropic introduced Agent Skills on 16 October 2025 in t...
   5  final    with-skill         Anthropic introduced Agent Skills on 16 October 2025 in t...

without skills: 4 model calls, ~

The trace shows the extra step at turn 1 (`load_skill`), and the token counts show what it cost. That is the price of a procedure the model can follow reliably; in notebook 03 an eval suite measures what it buys.

### 4.3 Skills, tools and prompts do different jobs

| | Gives the model | Lives in | Loaded | Example in this module |
|---|---|---|---|---|
| **Prompt** | the task, and standing rules | the user message and the system prompt | every call | the `research` role prompt |
| **Tool** | an ability | a function with a name, a description and a schema | described every call, run on request | `search_docs`, `check_citation` |
| **Skill** | a procedure: how to use the abilities well for a kind of work | a folder with `SKILL.md` and optional files | name and description every call, body on demand | `cite-sources` |

A prompt without tools cannot act. Tools without a procedure get used badly. A skill without tools has nothing to instruct. The harness is what puts the three together (Anthropic, "Equipping agents for the real world with Agent Skills", 16 Oct 2025).

## 5. Switching engines

Everything in this notebook ran through one call, `model_for(role)`, and that is the whole switch.

- **To run on Claude:** put `ANTHROPIC_API_KEY=...` in a `.env` file at the repository root (copy `.env.example`). `describe_runtime()` at the top confirms which engine is active. `HARNESS_MODEL` and `HARNESS_WORKER_MODEL` in the same file override the default model ids.
- **To run on the stand-in:** leave the key out, or pass `real=False` to `model_for` for one agent, as the reference runs do.
- **Custom stand-ins stay stand-ins.** The hostile `hacker` policy and the `skilled_policy` above are `ScriptedModel`s on purpose: they imitate one behaviour we want to see every time, and no key changes that.
- **Reading traces on two engines.** Same format either way. On the stand-in the token column is an estimate (about four characters per token); on Claude it is the real input size of each call as reported by the API.

In [23]:
from harness import describe_runtime

print(describe_runtime())
print("research engine today:", type(model_for("research")[0]).__name__, "|",
      "forced stand-in:", type(model_for("research", real=False)[0]).__name__)

Engine: real Claude models - claude-opus-5 for lead agents, claude-sonnet-5 for workers and judges (key loaded from .env; the key is never printed).
research engine today: AnthropicModel | forced stand-in: ScriptedModel


## 6. Key takeaways

- **The model is one method.** Everything a harness needs from a model is `complete(system, messages, tools) -> Message`. A stand-in and a Claude model are drop-ins for one another; `model_for(role)` picks one by tier, and you choose by capability, cost and latency.
- **Context is a budget.** Every tool result stays in the conversation for every later call. Compaction folds the old part into a note when the budget is hit; structured notes move the facts that matter out of the window altogether.
- **A tool is its description.** The model reads a name, a sentence and a JSON schema, nothing else. Vague descriptions waste turns; unused tools cost tokens on every call; error messages must tell the model what to try next.
- **Permissions are decided outside the model.** Risk levels plus a permission rule stop a dangerous tool before it runs, whatever the model asked for. Rules in code beat prompts humans click through.
- **MCP is the standard plug.** Any tool server fits any model client; a tool over MCP is still a name, a description and a schema.
- **Skills are procedures, disclosed progressively.** Names and descriptions cost a few dozen tokens per skill; the body loads only when a task matches; supporting files only while executing. Prompts give the task, tools give abilities, skills give procedures.

## 7. Practice exercises

Try these yourself before peeking at the solutions.

**Exercise 1 - A fourth skill.** Write a new skill folder under `outputs/my-skills/` with a `SKILL.md` that follows the same layout as `cite-sources` (front matter with `name` and `description`, then numbered steps). A good candidate: `compare-dates`, a procedure for putting two dated events from the library in order. Load it with `SkillIndex("outputs/my-skills")`, print the catalog and its token count, then load the body. How many tokens does your skill add to every call at level 1?

**Exercise 2 - Tune the budget.** In section 2 the first compaction happened during the fourth question. Find a `budget_tokens` value for which the first compaction happens during the **third** question instead, and check that the answers stay cited. Hint: write a loop over a few candidate budgets and record on which question the compaction count first becomes non-zero.

**Exercise 3 - A better description.** Rewrite the vague `lookup` tool from section 3 with the `@tool` decorator and a proper docstring: say what is searched, what the argument should contain, what comes back and what to do next. Print its schema next to the old one and compare the token cost of the two descriptions.

**Exercise 4 - Errors as interface.** Write a tool `release_date(product)` that knows the release dates of three things from this module (MCP, Agent Skills, harness engineering as a named discipline) and, for an unknown product, returns a helpful error message listing the products it knows instead of raising an exception. Call it once with a known product and once with an unknown one.

## 8. Solutions

**Solution 1 - A fourth skill.** The folder name, the `name` in the front matter and the description are all the agent sees at level 1. Keep the description to one line that says when the skill applies; that line is what a model matches against the task.

In [24]:
from pathlib import Path

skill_dir = Path("outputs/my-skills/compare-dates")
skill_dir.mkdir(parents=True, exist_ok=True)
(skill_dir / "SKILL.md").write_text(
    "---\n"
    "name: compare-dates\n"
    "description: Put two dated events from the course library in order and say how far apart they are.\n"
    "---\n"
    "1. Look up each event with search_docs and read the matching document with read_doc.\n"
    "2. Quote the date of each event and tag each with its [source: id].\n"
    "3. State which came first and the number of months between them.\n",
    encoding="utf-8")

mine = SkillIndex("outputs/my-skills")
print(mine.catalog())
print(f"level 1 cost of this skill: {mine.catalog_tokens()} tokens on every call\n")
print(mine.load("compare-dates"))

- compare-dates: Put two dated events from the course library in order and say how far apart they are.
level 1 cost of this skill: 25 tokens on every call

# Skill: compare-dates

1. Look up each event with search_docs and read the matching document with read_doc.
2. Quote the date of each event and tag each with its [source: id].
3. State which came first and the number of months between them.


**Solution 2 - Tune the budget.** This loop runs five questions for each of five budgets, so it stays on the stand-in (`real=False`): twenty-five agent runs for free and with the same result every time. In section 2 the window reached just over 2,100 tokens by the end of the third question. Any budget below that, but large enough for one question's worth of tool results, makes the window cross the line during the third run. The loop below tries a few budgets and reports where the first compaction lands; smaller budgets compact more often, but every answer stays cited, because the current task and its fresh tool results are never summarized.

In [25]:
def first_compaction(budget):
    agent = Agent(model_for("research", real=False)[0], tools=[search_docs, read_doc, check_citation],
                  system="Answer from the library and cite.",           # the stand-in: 25 free, repeatable runs
                  context=ContextWindow(budget_tokens=budget), name="tuned")
    history, first, all_cited = [], None, True
    for i, q in enumerate(QUESTIONS, 1):
        result = agent.run(q, history=history)
        history = result.messages
        all_cited = all_cited and "[source:" in result.answer
        if first is None and agent.context.compactions:
            first = i
    return first, agent.context.compactions, all_cited


for budget in (3000, 2500, 2200, 2000, 1500):
    first, total, cited = first_compaction(budget)
    print(f"budget {budget:>5}: first compaction on question {first}, {total} compaction(s) in total, all answers cited={cited}")

budget  3000: first compaction on question 5, 1 compaction(s) in total, all answers cited=True
budget  2500: first compaction on question 4, 1 compaction(s) in total, all answers cited=True


budget  2200: first compaction on question 4, 2 compaction(s) in total, all answers cited=True


budget  2000: first compaction on question 3, 2 compaction(s) in total, all answers cited=True


budget  1500: first compaction on question 3, 3 compaction(s) in total, all answers cited=True


**Solution 3 - A better description.** The docstring's first paragraph becomes the description and the `q: ...` line documents the argument. The better tool costs more tokens to describe, and buys a model that knows what to send and what to do with the result.

In [26]:
@tool(name="lookup_library")
def lookup_library(q: str, k: int = 3) -> str:
    """Search the course library (ten short documents about harness engineering) for a topic. Returns JSON with the id, title and a snippet of the best matches; pass an id to read_doc to get the full text.

    q: two to six key terms, for example "model context protocol release"
    k: how many matches to return, default 3
    """
    return search_docs.fn(q, k)


for t in (lookup, lookup_library):
    print(f"--- {t.name}: {estimate_tokens(t.description)} tokens of description")
    print(json.dumps(t.schema(), indent=2))
    print()

--- lookup: 2 tokens of description
{
  "name": "lookup",
  "description": "Searches.",
  "parameters": {
    "type": "object",
    "properties": {
      "q": {
        "type": "string"
      }
    },
    "required": [
      "q"
    ]
  }
}

--- lookup_library: 49 tokens of description
{
  "name": "lookup_library",
  "description": "Search the course library (ten short documents about harness engineering) for a topic. Returns JSON with the id, title and a snippet of the best matches; pass an id to read_doc to get the full text.",
  "parameters": {
    "type": "object",
    "properties": {
      "q": {
        "type": "string",
        "description": "two to six key terms, for example \"model context protocol release\""
      },
      "k": {
        "type": "integer",
        "default": 3,
        "description": "how many matches to return, default 3"
      }
    },
    "required": [
      "q"
    ]
  }
}



**Solution 4 - Errors as interface.** The tool looks the product up in a small dictionary. For an unknown name it does not raise; it returns a sentence that names the problem and lists the valid options, so a model can retry with a name that exists.

In [27]:
RELEASES = {
    "mcp": "25 November 2024 (Anthropic open-sources the Model Context Protocol)",
    "agent skills": "16 October 2025 (Anthropic introduces Agent Skills)",
    "harness engineering": "11 February 2026 (OpenAI names harness engineering as a discipline)",
}


@tool
def release_date(product: str) -> str:
    """Return the release date of one product or concept covered in this module.

    product: one of the known product names, case does not matter
    """
    key = product.strip().lower()
    if key not in RELEASES:
        return (f"Error: unknown product '{product}'. Known products: {', '.join(RELEASES)}. "
                f"Call release_date again with one of them.")
    return RELEASES[key]


print(release_date.call({"product": "MCP"}))
print(release_date.call({"product": "Codex"}))

25 November 2024 (Anthropic open-sources the Model Context Protocol)
Error: unknown product 'Codex'. Known products: mcp, agent skills, harness engineering. Call release_date again with one of them.
